# WA1 Water Footprint Model -- Manager Notebook

Orchestrates training of the Cross-Attention Geo-Aware Water Footprint Network.
All model logic lives in `src/` -- this notebook only imports and calls.

## 1. Initialization

In [ ]:
# -- Configuration --
GITHUB_USERNAME = ""  # Fill in
GITHUB_TOKEN = ""     # Fill in (fine-grained PAT)
REPO_URL = ""         # Fill in

import os
IN_COLAB = "COLAB_GPU" in os.environ or os.path.exists("/content")

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_BASE = "/content/drive/MyDrive/ESPResso-V2"
    os.makedirs(DRIVE_BASE, exist_ok=True)
    print(f"Drive mounted at {DRIVE_BASE}")
else:
    DRIVE_BASE = None
    print("Running locally")

## 2. Clone / Update Repository

In [ ]:
if IN_COLAB:
    REPO_DIR = "/content/ESPResso-V2"
    if not os.path.exists(REPO_DIR):
        auth_url = REPO_URL.replace("https://", f"https://{GITHUB_USERNAME}:{GITHUB_TOKEN}@")
        !GIT_LFS_SKIP_SMUDGE=1 git clone {auth_url} {REPO_DIR}
    else:
        !cd {REPO_DIR} && git pull
    os.chdir(REPO_DIR)
else:
    REPO_DIR = os.getcwd()

import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"Working directory: {os.getcwd()}")

## 3. Check Data Cache

In [ ]:
DATA_PATH = os.path.join(REPO_DIR, "model", "data", "water_footprint.csv")

if IN_COLAB:
    CACHE_PATH = os.path.join(DRIVE_BASE, "data", "water_footprint.csv")
    if os.path.exists(CACHE_PATH) and not os.path.exists(DATA_PATH):
        os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
        !cp {CACHE_PATH} {DATA_PATH}
        print(f"Restored data from Drive cache")
    elif not os.path.exists(DATA_PATH):
        print("Data not found. Run: git lfs pull --include='model/data/**'")
        !cd {REPO_DIR} && git lfs pull --include="model/data/**"
        if os.path.exists(DATA_PATH):
            os.makedirs(os.path.dirname(CACHE_PATH), exist_ok=True)
            !cp {DATA_PATH} {CACHE_PATH}
            print("Cached data to Drive")

assert os.path.exists(DATA_PATH), f"Data not found at {DATA_PATH}"
print(f"Data ready: {DATA_PATH}")

## 4. Smoke Test

In [ ]:
from model.water_footprint.src.utils.config import WA1Config, set_seed
from model.water_footprint.src.training.checkpoint import smoke_test

config = WA1Config()

if IN_COLAB:
    config.checkpoint_dir = os.path.join(DRIVE_BASE, "checkpoints", "water_footprint")

result = smoke_test(config)
assert result["passed"], f"Smoke test failed: {result}"
print(f"Smoke test passed -- train_loss={result['train_loss']:.4f}, gate_A={result['gate_a']:.4f}, gate_F={result['gate_f']:.4f}")

## 5. Check for Existing Checkpoint

In [ ]:
import glob

checkpoint_pattern = os.path.join(config.checkpoint_dir, "checkpoint_*.pt")
existing = sorted(glob.glob(checkpoint_pattern))

if existing:
    print(f"Found {len(existing)} existing checkpoint(s):")
    for cp in existing:
        print(f"  {cp}")
    RESUME_FROM = existing[-1]
    print(f"Will resume from: {RESUME_FROM}")
else:
    RESUME_FROM = None
    print("No existing checkpoints. Training from scratch.")

## 6. Prepare Data

In [ ]:
import torch
from torch.utils.data import DataLoader
from model.water_footprint.src.preprocessing.dataset import WaterFootprintDataset
from model.water_footprint.src.preprocessing.transforms import (
    Log1pZScoreTransform, create_splits,
)

set_seed(config.seed)
device = config.device
print(f"Device: {device}")

# Load dataset and split
full_dataset = WaterFootprintDataset(config.data_path)
train_set, val_set, test_set = create_splits(
    full_dataset,
    seed=config.seed,
    train_ratio=config.train_ratio,
    val_ratio=config.val_ratio,
)
print(f"Split: train={len(train_set)}, val={len(val_set)}, test={len(test_set)}")

# Fit transform on training targets
transform = Log1pZScoreTransform()
train_targets = torch.stack([
    torch.tensor([train_set[i]["wf_raw"], train_set[i]["wf_processing"], train_set[i]["wf_packaging"]])
    for i in range(len(train_set))
])
transform.fit(train_targets)
print(f"Transform fitted: {transform.state_dict()}")

# DataLoaders
NUM_WORKERS = 4 if IN_COLAB else 0
train_loader = DataLoader(
    train_set, batch_size=config.batch_size, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)
val_loader = DataLoader(
    val_set, batch_size=config.batch_size, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)
test_loader = DataLoader(
    test_set, batch_size=config.batch_size, shuffle=False,
)
print(f"Loaders ready: {len(train_loader)} train batches, {len(val_loader)} val batches")

## 7. Train

In [ ]:
from model.water_footprint.src.training.model import WA1Model
from model.water_footprint.src.training.loss import UWSOLoss
from model.water_footprint.src.training.trainer import WA1Trainer

# Build model
model = WA1Model(config).to(device)
loss_fn = UWSOLoss(config).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model: {n_params:,} parameters")

# Build trainer
trainer = WA1Trainer(
    config=config,
    model=model,
    loss_fn=loss_fn,
    train_loader=train_loader,
    val_loader=val_loader,
    transform=transform,
    device=device,
)

# Resume if checkpoint exists
if RESUME_FROM:
    trainer.load_checkpoint(RESUME_FROM)
    print(f"Resumed from epoch {trainer.start_epoch}")

# Viability check (first 5 epochs)
viable = trainer.viability_check(n_canary=config.viability_canary_epochs)
assert viable, "Viability check failed -- see diagnostics above"
print("Viability check passed. Starting full training...")

# Full training
result = trainer.train(
    max_epochs=config.max_epochs,
    patience=config.patience,
)
print(f"Training complete. Best val loss: {result['best_val_loss']:.4f} at epoch {result['best_epoch']}")

## 8. Evaluate

In [ ]:
from model.water_footprint.src.evaluation.metrics import compute_metrics, per_tier_evaluation

# Load best checkpoint
best_ckpt = sorted(glob.glob(checkpoint_pattern))[-1]
trainer.load_checkpoint(best_ckpt)
print(f"Loaded best checkpoint: {best_ckpt}")

# Test set evaluation (Tier F -- full data)
print("\n--- Test Set (Tier F) ---")
test_metrics = trainer.val_epoch(loader=test_loader, tier="F")
for k, v in test_metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

# Per-tier evaluation matrix
print("\n--- Per-Tier Evaluation (Validation Set) ---")
tier_matrix = per_tier_evaluation(
    model=model,
    dataloader=val_loader,
    transform=transform,
    tiers=["A", "B", "C", "D", "E", "F"],
    device=device,
)
print(tier_matrix.to_string())

## 9. Summary

In [ ]:
print("=" * 60)
print("WA1 Water Footprint Model -- Training Summary")
print("=" * 60)
print(f"Parameters: {n_params:,}")
print(f"Best epoch: {result['best_epoch']}")
print(f"Best val loss: {result['best_val_loss']:.4f}")
print(f"Device: {device}")
print(f"Checkpoint: {config.checkpoint_dir}")
print(f"Experiment log: {os.path.join(config.checkpoint_dir, 'runs.jsonl')}")